In [1]:
from dotenv import load_dotenv 
load_dotenv()

True

In [2]:
import os 
from getpass import getpass 

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [4]:
#Quick illustration of the two approaches deterministic and model based

import re 

#----- Deterministic approach -----
def deterministic_guardrail(text:str) -> bool:
    """ Returns True if content is blocked. """
    banned_keywords = ["hack", "exploit", "malware", "bomb"]

    return any( kw in text.lower() for kw in banned_keywords )

test_inputs = [
    "How do i hack into a database?",
    "What is the captial of France",
    "Explain how malware spreads",
]

print("=== Deterministic Guradrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "Allowed"
    print(f"{status}: {inp}")


=== Deterministic Guradrail Demo ===
BLOCKED: How do i hack into a database?
Allowed: What is the captial of France
BLOCKED: Explain how malware spreads


In [5]:
from langchain_openai import ChatOpenAI 

# -- Model based approach --- 
def model_based_guardrail(text:str) -> str:
    """ Uses an LLM to evaluate content saftey. Returns SAFE or UNSAFE."""
    model = ChatOpenAI(model = "gpt-4o-mini", temperature=0)
    prompt = f"""Is the following user input safe to process?
    Reply with only 'SAFE' or 'UNSAFE'.

    Input: {text}"""
    result = model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("===Model Based Guardrails Demo ===")
for inp in test_inputs:
     verdict = model_based_guardrail(inp)
     status = "UNSAFE " if "UNSAFE" in verdict else "SAFE"
     print(f"{status}: {inp}")

===Model Based Guardrails Demo ===
UNSAFE : How do i hack into a database?
SAFE: What is the captial of France
SAFE: Explain how malware spreads


In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI 
from langchain_core.tools import tool 

#Define a simple dummy tool 
@tool 
def customer_lookup(query:str) -> str:
    """Look up customer information"""
    return f"Customer record found for query: {query}"

#Create agent with PII Middleware 
agent = create_agent(
    model = "gpt-4o",
    tools = [customer_lookup],
    middleware = [
        PIIMiddleware(
            "email",
            strategy = "redact",
            apply_to_input=True,
        ),

        #Mask credit cards in user input 
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        #Block API Keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector = r"sk-[a-zA-Z0-9]{32}",
            strategy = "block",
            apply_to_input = True,
        ),
    ],
)

print("Agent with PII moddleware created successfully!")

Agent with PII moddleware created successfully!


In [8]:
#Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role":"user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I found customer records for both the email address and the card number you provided. How can I assist you further with your account?


In [9]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='aaa86bf2-941b-4a4a-83ca-82c9181bb6a5'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 70, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_40aced2ba4', 'id': 'chatcmpl-EPdLT1zcZrfFafWGqNa4Z5IjNIJnq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0b717-41b8-77b3-86d5-05c8f57ad5f5-0', tool_calls=[{'name': 'customer_lookup', 'args': {'q

In [10]:
#Test API Key Blocking
try:
    result = agent.invoke({
        "messages":[{
            "role":"user",
            "content":"Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"Blocked as expected: {e}")


Blocked as expected: Detected 1 instance(s) of api_key in text content


In [37]:
#Human in the loop, pauses agent execution before sensitive operations and waits for human approval.

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver 
from langgraph.types import Command 
from langchain_core.tools import tool 

@tool 
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search result for : {query}"

@tool 
def send_email(to: str, subject:str, body:str) -> str:
    """Send an email to a recepient"""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table:str, condition:str) ->str:
    """Delete records from the database"""
    return f"Deleted records from {table} where {condition}"

#Create agent with HITL middleware
hitl_agent = create_agent(
    model="gpt-4o",
    tools=[search_web,send_email,delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email":True,   #Require Approval
                "delete_records": True,  #Require Approval
                "search_web": False, #Auto Approval 
            }
        ),
    ],
    checkpointer = InMemorySaver(), 
)

print("Human-in-the-Loop- agent created! ")

Human-in-the-Loop- agent created! 


In [38]:
#Step 1: Invoke - agent will pause before send_email 
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages":[{"role": "user", "content":"Send an email to team@company.com about Q4 results"}]},
    config = config
)

print("===Agent paused - awaiting human approval ===")
print(result)

===Agent paused - awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about Q4 results', additional_kwargs={}, response_metadata={}, id='f731c3bc-0760-43f8-aee4-3cef953b02b0'), AIMessage(content='What would you like the email to say?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 110, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_087f6ae320', 'id': 'chatcmpl-EPsRPrtV4hMO0ezkXexG8upCZFkie', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ba8c-b576-7fa0-86a3-8378f39538a2-0', tool_c

In [39]:
#Step 2: Human reviews and APPROVES 
approved_result = hitl_agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)


=== Approved! Final response ===
What would you like the email to say?


In [40]:
#step3: Alternatives - Human Rejects
config2 = {"configurable": {"thread_id":"session_002"}}

hitl_agent.invoke(
    {
        "messages": [
            {
                "role":"user",
                "content":"Delete all records from the users tables where active=false"
             }
        ]
    },
    config = config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("===Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

===Rejected! Final response ===
It seems like you decided not to proceed with the deletion of records. If you need any further assistance or have any other requests, just let me know!


In [43]:
#Custom Guardrail 
from typing import Any 
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime 
from langchain.agents import create_agent
from langchain_core.tools import tool 

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent process anything - zero LLM cost for blocked requests.
    
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self,state: AgentState, runtime:Runtime)-> dict[str, Any] | None:
        if not state["messages"]:
            return None 
        first_message = state["messages"][0]
        if first_message.type != "human":
            return None 

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked - keyword detected: '{keyword}' ")
                return {
                    "messages":[{
                        "role": "assistant",
                        "content":(
                            "I cannot process requests containing inappropriate content."
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
            return None 

@tool 
def search_tool(query: str) ->str:
    """Search for information."""
    return f"Results for: {query}"

#Create agent with contnent filter
filtered_agent = create_agent(
    model="gpt-4o",
    tools = [search_tool],
    middleware = [
        ContentFilterMiddleware(
        banned_keywords = ["hack", "exploit","malware","jailbreak","bypass"]
            ),

        ],
)

print("Content filter agent created")




Content filter agent created


In [44]:
#Test 1: Safe request  should pass through 
result = filtered_agent.invoke({
    "messages":[{"role": "user", "content":"What is machine learning?"}]
})

print("Safe request response")
print(result["messages"][-1].content)

Safe request response
Machine learning is a branch of artificial intelligence (AI) that focuses on developing algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data. Unlike traditional programming, where explicit instructions are provided to achieve a specific outcome, machine learning involves training a model on a dataset to recognize patterns, relationships, or trends. This model can then be used to predict or classify new data. Machine learning is widely used in various applications, such as recommendation systems, image and speech recognition, fraud detection, and autonomous systems.


In [45]:
result = filtered_agent.invoke({
    "messages":[{"role":"user","content":"How do i hack into a server?"}]

})

print("Unsafe request response")
print(result["messages"][-1].content)


Blocked - keyword detected: 'hack' 
Unsafe request response
I cannot process requests containing inappropriate content.Please rephrase your request.


In [50]:
#Custom Guardrails after agent hook 

from typing import Any 
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime 
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool 

class SafteyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response saftey.
    Runs AFTER the agents produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        self.saftey_model = ChatOpenAI(model="gpt-4o-mini",temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self,state: AgentState, runtime: Runtime) ->dict[str, Any] | None: 
        if not state["messages"]:
            return None 

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None 

        #Use a lightweight model as a saftey judge
        saftey_prompt =f""" Evaluate if this AI response is safe and appropriate for users. 
        Respond with only 'SAFE' or 'UNSAFE'.
        """ 
        result = self.saftey_model.invoke([{"role":"user", "content": saftey_prompt}])

        if "UNSAFE" in result.content.upper():
            print(" Output flagged as UNSAFE - replacing with  safe fallback")

            last_message.content = (
                "I'm unable to provide that response"
                "Please rephrase your request or contact support. "
            )

@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"

safe_agent = create_agent(
    model ="gpt-4o",
    tools = [general_tool],
    middleware = [SafteyGuardrailMiddleware()],
)

print("Output saftey agent created!")

Output saftey agent created!


In [51]:
#Test output saftey check 
result = safe_agent.invoke({
    "messages":[{"role":"user", "content":"What is the weather like today?"}]
})

print("Response:")
print(result["messages"][-1].content)

Response:
I can't provide real-time weather data. You can check your local weather forecast or a weather application for the most up-to-date information.


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool 

@tool
def search_tool(query: str)-> str:
    """Search for information."""
    return f"Search result: {query}"

@tool 
def send_email_tool(to: str, body:str) ->str:
    """Send an email."""
    return f"Email send to {to}"

#Full layered guardrail stack
production_agent = create_agent(
    model = "gpt-4o",
    tools = [search_tool, send_email_tool],
    middleware=[
        #Layer1: Deterministic input (before agent)
        ContentFilterMiddleware(banned_keywords=["hack","exploit","malware"]),

        #Layer2 : PII redaction on input 
        PIIMiddleware("email",strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask",apply_to_input=True),

        #Layer3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool":False}
        ),

        #Layer4: PII redaction on output 
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        #Layer5: Model based output saftey
        SafteyGuardrailMiddleware(),

    ],
    checkpointer = InMemorySaver(),
)


print("Production grade agent with 5 layer guardrails created!")